## NAME : THIRUMALAI K
## REG_NO: 212224240176

In [1]:
%pip install -U langchain langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu pypdf sentence-transformers ipywidgets

  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.8
    Uninstalling langchain-1.4.8:
      Successfully uninstalled langchain-1.4.8
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import torch
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, Markdown, clear_output
from getpass import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [4]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

print("Groq API Key configured successfully!")

Enter your Groq API Key:  ········


Groq API Key configured successfully!


In [5]:
upload_button = widgets.FileUpload(
    accept=".pdf",
    multiple=False,
    description="Upload PDF"
)

display(upload_button)

FileUpload(value=(), accept='.pdf', description='Upload PDF')

In [6]:
if not upload_button.value:
    raise ValueError("Please upload a PDF document first.")

uploaded_file = upload_button.value

# Support ipywidgets 7 and 8
if isinstance(uploaded_file, dict):
    file_name = list(uploaded_file.keys())[0]
    file_content = uploaded_file[file_name]["content"]
else:
    uploaded_file = uploaded_file[0]
    file_name = uploaded_file["name"]
    file_content = uploaded_file["content"]

pdf_path = Path(file_name)

with open(pdf_path, "wb") as f:
    f.write(file_content)

print("PDF uploaded successfully!")
print("File Name:", file_name)
print("File Size:", round(len(file_content) / 1024, 2), "KB")

PDF uploaded successfully!
File Name: AI_Introduction.pdf
File Size: 467.48 KB


In [7]:
loader = PyPDFLoader(str(pdf_path))

documents = loader.load()

print("PDF Loaded Successfully!")
print("Total Pages:", len(documents))

for i, doc in enumerate(documents[:3]):
    print(f"\nPage {i + 1}:")
    print(doc.page_content[:300])

PDF Loaded Successfully!
Total Pages: 2

Page 1:
INTRODUCTION TO ARTIFICIAL INTELLIGENCE 
1. ArƟﬁcial Intelligence 
ArƟﬁcial Intelligence (AI) is a branch of computer science that focuses on developing machines 
capable of performing tasks that normally require human intelligence. 
AI systems can learn, solve problems, understand language, recogni

Page 2:
 Agriculture: Crop monitoring and disease detecƟon. 
6. GeneraƟve AI 
GeneraƟve AI is a type of ArƟﬁcial Intelligence that can generate new content such as text, 
images, audio, and code. 
Examples include ChatGPT, Google Gemini, and other generaƟve AI applicaƟons. 
7. Conclusion 
ArƟﬁcial Intellig


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Text Splitting Completed!")
print("Total Chunks:", len(chunks))

Text Splitting Completed!
Total Chunks: 3


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device}
)

print("Embeddings Model Loaded Successfully!")

Using Device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Model Loaded Successfully!


In [10]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS Vector Database Created Successfully!")

FAISS Vector Database Created Successfully!


In [11]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever Configured Successfully!")

Retriever Configured Successfully!


In [12]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Groq LLM Initialized Successfully!")

Groq LLM Initialized Successfully!


In [13]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful PDF Question Answering Assistant.

Answer the question using ONLY the information
provided in the context below.

If the answer is not available in the document,
clearly say:

"The answer is not available in the uploaded PDF."

Do not create imaginary information.

Context:
{context}

Question:
{question}

Answer:
""")

print("Prompt Template Created Successfully!")

Prompt Template Created Successfully!


In [14]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Question Answering Chain Created Successfully!")

RAG Question Answering Chain Created Successfully!


In [15]:
def ask_pdf(question):

    relevant_docs = retriever.invoke(question)

    answer = rag_chain.invoke(question)

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCE PAGES:")

    source_pages = sorted(set(
        doc.metadata.get("page", 0) + 1
        for doc in relevant_docs
    ))

    print(source_pages)

    return answer

In [16]:
question = input("Enter your question: ")

answer = ask_pdf(question)

Enter your question:  What is the main topic of this document?



QUESTION:
What is the main topic of this document?

ANSWER:
The main topic of this document is **Artificial Intelligence (AI)**.

SOURCE PAGES:
[1, 2]


In [ ]:
print("PDF QUESTION ANSWERING CHATBOT")
print("Type 'exit' to stop the chatbot.")

while True:

    question = input("\nYou: ")

    if question.lower().strip() == "exit":
        print("Chatbot session ended.")
        break

    if not question.strip():
        print("Please enter a valid question.")
        continue

    try:
        answer = ask_pdf(question)

    except Exception as e:
        print("Error:", e)

PDF QUESTION ANSWERING CHATBOT
Type 'exit' to stop the chatbot.



You:  Explain the important concepts mentioned in the document.



QUESTION:
Explain the important concepts mentioned in the document.

ANSWER:
**Important concepts mentioned in the document**

1. **Artificial Intelligence (AI)**  
   - A branch of computer science that creates machines capable of performing tasks that normally require human intelligence, such as learning, problem‑solving, language understanding, image recognition, and decision‑making.

2. **Machine Learning (ML)**  
   - A subset of AI that enables computers to learn from data and improve performance without being explicitly programmed for each task.  
   - **Three main types of ML:**  
     • **Supervised Learning** – learning from labeled data.  
     • **Unsupervised Learning** – discovering patterns in unlabeled data.  
     • **Reinforcement Learning** – learning through trial‑and‑error interactions with an environment.

3. **Deep Learning**  
   - A further subset of ML that uses artificial neural networks to learn complex patterns from large amounts of data.  
   - Commonly a


You:  What are the main steps involved in the process?



QUESTION:
What are the main steps involved in the process?

ANSWER:
The answer is not available in the uploaded PDF.

SOURCE PAGES:
[1, 2]
